# 09. 서브그래프 — 그래프 안의 그래프

## 학습 목표

서브그래프로 복잡한 워크플로를 모듈화합니다.

## 9.1 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4.1")

In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 9.2 서브그래프 개념

- **서브그래프**: 다른 그래프의 노드로 사용되는 독립적인 그래프
- **장점**: 모듈화, 재사용, 팀별 독립 개발
- 각 서브그래프는 자체 상태(State)를 가짐
- 부모 <-> 서브그래프 간 상태는 **공유 키**로 매핑

## 9.3 서브그래프 만들기

In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
import operator

# === Subgraph 1: Text Analysis ===
class AnalysisState(TypedDict):
    text: str
    word_count: int
    char_count: int

def count_words(state: AnalysisState) -> dict:
    return {"word_count": len(state["text"].split())}

def count_chars(state: AnalysisState) -> dict:
    return {"char_count": len(state["text"])}

analysis_builder = StateGraph(AnalysisState)
analysis_builder.add_node("words", count_words)
analysis_builder.add_node("chars", count_chars)
analysis_builder.add_edge(START, "words")
analysis_builder.add_edge("words", "chars")
analysis_builder.add_edge("chars", END)
analysis_subgraph = analysis_builder.compile()

# Test subgraph independently
result = analysis_subgraph.invoke({"text": "LangGraph에서 보내는 인사"}, config=lf_config)
print(f"서브그래프 테스트: 단어={result['word_count']}, 문자={result['char_count']}")

서브그래프 테스트: 단어=3, 문자=18


## 9.4 부모 그래프에 서브그래프 추가

In [4]:
# === Parent Graph: Text Pipeline ===

class PipelineState(TypedDict):
    text: str
    word_count: int
    char_count: int
    summary: str


def create_summary(state: PipelineState) -> dict:
    return {
        "summary": f"텍스트는 {state['word_count']}개의 단어와 {state['char_count']}개의 문자를 포함합니다."
    }


parent_builder = StateGraph(PipelineState)

# Add subgraph as a node (shared keys: text, word_count, char_count)
parent_builder.add_node("analyze", analysis_subgraph)
parent_builder.add_node("summarize", create_summary)

parent_builder.add_edge(START, "analyze")
parent_builder.add_edge("analyze", "summarize")
parent_builder.add_edge("summarize", END)

pipeline = parent_builder.compile()

result = pipeline.invoke(
    {
        "text": "LangGraph는 강력한 상태 기반 에이전트 워크플로를 가능하게 합니다"
    },
    config=lf_config
)

print(f"요약: {result['summary']}")

요약: 텍스트는 8개의 단어와 40개의 문자를 포함합니다.


## 9.4.1 패턴 1: 래퍼 노드를 통한 서브그래프 호출 (상태 스키마가 다른 경우)

위 예시(9.4)에서는 부모 그래프와 서브그래프가 **동일한 키**(`text`, `word_count`, `char_count`)를 공유했기 때문에, 컴파일된 서브그래프를 `add_node()`에 직접 전달할 수 있었습니다.

실무에서는 부모 그래프와 서브그래프의 **상태 스키마가 완전히 다른** 경우가 많습니다. 이때는 **래퍼 함수(wrapper function)**를 사용하여:

1. 부모 상태에서 필요한 필드를 **추출**하여 서브그래프 입력으로 변환
2. 서브그래프를 **실행**
3. 서브그래프 출력을 부모 상태 형식으로 **매핑**

합니다. 공식 문서에서 **Pattern 1: Call Subgraph Inside a Node**라고 부르는 방식입니다.

In [5]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


# ── Step 1: 서브그래프 정의 (자체 상태 스키마: SubState) ──
class SubState(TypedDict):
    log_entry: str
    sentiment: str
    confidence: float


def analyze_sentiment(state: SubState) -> dict:
    """간단한 규칙 기반 감성 분석"""
    text = state["log_entry"].lower()

    positive_words = {
        "good", "great", "excellent", "happy", "love", "amazing", "wonderful"
    }

    negative_words = {
        "bad", "terrible", "awful", "hate", "poor", "horrible", "sad"
    }

    pos = sum(1 for w in text.split() if w in positive_words)
    neg = sum(1 for w in text.split() if w in negative_words)

    total = pos + neg

    if total == 0:
        return {"sentiment": "neutral", "confidence": 0.5}

    elif pos > neg:
        return {"sentiment": "positive", "confidence": round(pos / total, 2)}

    else:
        return {"sentiment": "negative", "confidence": round(neg / total, 2)}


sub_builder = StateGraph(SubState)

sub_builder.add_node("analyze_sentiment", analyze_sentiment)
sub_builder.add_edge(START, "analyze_sentiment")
sub_builder.add_edge("analyze_sentiment", END)

sentiment_subgraph = sub_builder.compile()


# 서브그래프 단독 테스트
sub_result = sentiment_subgraph.invoke(
    {"log_entry": "The product is great and amazing"},
    config=lf_config
)

print(
    f"[서브그래프 단독 테스트] sentiment={sub_result['sentiment']}, confidence={sub_result['confidence']}"
)


# ── Step 2: 부모 그래프 정의 (다른 상태 스키마: ParentState) ──
class ParentState(TypedDict):
    customer_name: str
    feedback_text: str
    analysis_result: str


# ── Step 3: 래퍼 함수 — 상태 변환의 핵심 ──
def call_sentiment_subgraph(state: ParentState) -> dict:
    """
    래퍼 함수가 하는 일:

    1) 부모 상태의 'feedback_text' → 서브그래프의 'log_entry'로 변환
    2) 서브그래프 실행
    3) 서브그래프 출력을 부모 상태의 'analysis_result'로 매핑
    """

    # Parent → Subgraph: 상태 변환
    subgraph_input = {
        "log_entry": state["feedback_text"]
    }

    # 서브그래프 실행
    subgraph_output = sentiment_subgraph.invoke(
        subgraph_input,
        config=lf_config
    )

    # Subgraph → Parent: 결과 매핑
    result_str = (
        f"{subgraph_output['sentiment']} "
        f"(confidence: {subgraph_output['confidence']})"
    )

    return {"analysis_result": result_str}


def generate_report(state: ParentState) -> dict:
    """최종 보고서 생성"""
    report = (
        f"[보고서] {state['customer_name']}: "
        f"{state['analysis_result']}"
    )

    return {"analysis_result": report}


# ── Step 4: 부모 그래프 구성 ──
parent_builder = StateGraph(ParentState)

parent_builder.add_node("sentiment_analysis", call_sentiment_subgraph)
parent_builder.add_node("report", generate_report)

parent_builder.add_edge(START, "sentiment_analysis")
parent_builder.add_edge("sentiment_analysis", "report")
parent_builder.add_edge("report", END)

parent_graph = parent_builder.compile()


# 실행
result = parent_graph.invoke(
    {
        "customer_name": "Alice",
        "feedback_text": "The product is great and the service was excellent",
    },
    config=lf_config
)

print(f"\n[최종 결과] {result['analysis_result']}")


# 다른 입력으로 추가 테스트
result2 = parent_graph.invoke(
    {
        "customer_name": "Bob",
        "feedback_text": "The experience was terrible and the quality is poor",
    },
    config=lf_config
)

print(f"[최종 결과] {result2['analysis_result']}")

[서브그래프 단독 테스트] sentiment=positive, confidence=1.0

[최종 결과] [보고서] Alice: positive (confidence: 1.0)
[최종 결과] [보고서] Bob: negative (confidence: 1.0)


## 9.5 LLM 기반 서브그래프

전문 에이전트를 서브그래프로 구성합니다.

In [6]:
from langchain.messages import HumanMessage, AnyMessage
from langgraph.graph import MessagesState

# Subgraph: Translation Agent
def translator(state: MessagesState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content="위 내용을 한국어로 번역해 주세요. 번역만 답해 주세요.")],
        config=lf_config,
)
    return {"messages": [response]}

translator_builder = StateGraph(MessagesState)
translator_builder.add_node("translate", translator)
translator_builder.add_edge(START, "translate")
translator_builder.add_edge("translate", END)
translator_subgraph = translator_builder.compile()

# Subgraph: Summarization Agent
def summarizer(state: MessagesState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content="위 내용을 한 문장으로 요약해 주세요.")],
        config=lf_config,
)
    return {"messages": [response]}

summarizer_builder = StateGraph(MessagesState)
summarizer_builder.add_node("summarize", summarizer)
summarizer_builder.add_edge(START, "summarize")
summarizer_builder.add_edge("summarize", END)
summarizer_subgraph = summarizer_builder.compile()

# Parent: Translate-then-Summarize Pipeline
parent_builder = StateGraph(MessagesState)
parent_builder.add_node("translator", translator_subgraph)
parent_builder.add_node("summarizer", summarizer_subgraph)
parent_builder.add_edge(START, "translator")
parent_builder.add_edge("translator", "summarizer")
parent_builder.add_edge("summarizer", END)

pipeline = parent_builder.compile()

result = pipeline.invoke({
    "messages": [HumanMessage(content="LangGraph는 LLM을 사용하여 상태 기반 멀티 액터 애플리케이션을 구축하는 프레임워크입니다.")]
}, config=lf_config)
print("최종 출력:", result["messages"][-1].content)

최종 출력: LangGraph는 LLM을 이용해 상태 기반 멀티 액터 애플리케이션을 만드는 프레임워크입니다.


## 9.6 서브그래프 스트리밍

서브그래프 내부 단계도 스트리밍 가능합니다.

In [7]:
print("서브그래프 스트리밍:")
for chunk in pipeline.stream(
    {"messages": [HumanMessage(content="Python은 범용 프로그래밍 언어입니다.")]},
    stream_mode="updates",
    subgraphs=True,
    config=lf_config,
):
    if isinstance(chunk, tuple) and len(chunk) == 2:
        ns, update = chunk
        for node, data in update.items():
            prefix = f"[{'/'.join(ns)}]" if ns else "[root]"
            print(f"  {prefix} {node}")

서브그래프 스트리밍:


  [translator:651b12a9-ad46-0b64-f7e6-5cf2b7b38770] translate
  [root] translator


  [summarizer:bbfdd16b-2f9f-fc55-66b4-77988fd50597] summarize
  [root] summarizer


## 9.7 요약

| 개념 | 설명 |
|------|------|
| 서브그래프 | 독립적으로 컴파일된 그래프를 노드로 사용 |
| 공유 키 | 부모-서브그래프 간 상태 매핑 |
| 모듈화 | 복잡한 워크플로를 작은 단위로 분리 |
| 스트리밍 | `subgraphs=True`로 내부 단계 추적 |

### 다음 단계
→ **[10_production.ipynb](./10_production.ipynb)**: 프로덕션 배포를 배웁니다.
